# DESCRIPTION (inspired by Leetcode.com)
Given an integer array nums and an integer k, write a function to identify the highest possible sum of a subarray within nums, where the subarray meets the following criteria: its length is k, and all of its elements are unique. If no such subarray exists, return 0.

Example 1: 

Input:

nums = [3, 2, 2, 3, 4, 6, 7, 7, -1]
k = 4

Output:

20

Explanation: The subarrays of nums with length 4 are:

[3, 2, 2, 3] # elements 3 and 2 are repeated.
[2, 2, 3, 4] # element 2 is repeated.
[2, 3, 4, 6] # meets the requirements and has a sum of 15.
[3, 4, 6, 7] # meets the requirements and has a sum of 20.
[4, 6, 7, 7] # element 7 is repeated.
[6, 7, 7, -1] # element 7 is repeated.
We return 20 because it is the maximum subarray sum of all the subarrays that meet the conditions.

Example 2: 

Input:

nums = [5, 5, 5, 5, 5]
k = 3

Output:

0

Explanation: Every subarray of length 3 contains duplicate elements, so no valid subarray exists. Return 0.

# Solution

I will use sliding window to pass through the whole array to find the subarray that meets the required criteria. To do that, I will need a variable to store the max sum, a dictionary to store occurence of each number in the subarray and a repeating counter to track how many numbers are repeated. 

If I repeatly scanning each subarary for meeting the criteria, the time complexity will be O(n^2). As I slide the window by adding the and removing numbers, I don't need to recheck numbers that I have already done so in previous steps. Thus, with a dictionary and counter to keeping track of occurence and duplication, I can achieve O(n).  

In [ ]:
class Solution_V1:
    def maxSum(self, nums: list[int], k: int) -> int:
        if len(nums) < k:
            return 0
       
        subarray_sum, max_sum = 0, float('-inf')
        start = 0
        repeating_count = 0
        subarray_dict = dict()
        
        for end in range(len(nums)):
           subarray_sum += nums[end]
           if nums[end] in subarray_dict and subarray_dict[nums[end]] > 0:
               subarray_dict[nums[end]] += 1
               repeating_count += 1
           else:
               subarray_dict[nums[end]] = 1
               
           if end-start+1 == k:
               if repeating_count == 0:    
                    max_sum = max(max_sum, subarray_sum)
                
               # slide window and update dict
               subarray_sum -= nums[start]
               subarray_dict[nums[start]] -= 1
               if subarray_dict[nums[start]] > 0: 
                   repeating_count -= 1
               start += 1

        return max_sum if max_sum > float('-inf') else 0

# FEEDBACK

Good job — your solution is correct and passes the tests.

A couple of interview-quality notes:

Your sliding window logic is solid, but repeating_count is a bit error-prone. It tracks only “extra duplicates,” so it’s harder to reason about than checking whether the window has exactly k distinct elements.
In the shrink step, you decrement the count for nums[start] but never remove keys when they hit 0. That works here, but it leaves stale entries in the dict and makes the invariant less clean.
len(nums) < k returning 0 is fine for this problem, but you should be aware that the final 0 return also means “no valid window found.”
For interviews, explain the window invariant clearly: sum + frequency map + distinct check.

In [48]:
class Solution:
    def maxSum(self, nums: list[int], k: int) -> int:
       if len(nums) < k:
           return 0
       
       subarray_sum, max_sum = 0, float('-inf')
       start = 0
       subarray_dict = dict()
        
       for end in range(len(nums)):
           subarray_sum += nums[end]
           subarray_dict[nums[end]] = subarray_dict.get(nums[end],0) + 1
               
           if end-start+1 == k:
               if len(subarray_dict) == k:    
                    max_sum = max(max_sum, subarray_sum)
                
               # slide window and update dict
               subarray_sum -= nums[start]
               subarray_dict[nums[start]] -= 1
               if subarray_dict[nums[start]] == 0: 
                   del subarray_dict[nums[start]]
               start += 1

       return 0 if max_sum == float('-inf') else max_sum

# FEEDBACK 2
Good job — your solution is correct and matches the expected sliding-window approach.

A couple of interview-level notes:

Your len(subarray_dict) == k check is the key insight: for a window of size k, having k distinct values means all elements are unique.
Time complexity is O(n) and space is O(k), which is optimal.
Small improvements:

You can simplify the frequency update with dict.get(...) for readability.
return max_sum if max_sum > float('-inf') else 0 works, but == float('-inf') is a bit clearer since you only care whether you ever updated it.
In an interview, mention that this assumes non-negative result fallback to 0 is intended by the problem statement; otherwise you’d return the actual maximum even if negative.

In [2]:
from dataclasses import dataclass
from typing import Callable

@dataclass(frozen=True)
class Input:
    nums: list[int]
    k: int

@dataclass(frozen=True)
class Test:
    input: Input
    expected_result: int

def run_tests(tests: list[Test], func: Callable[[list[int], int], int]):
    for test in tests:
        result = func(test.input.nums, test.input.k)
        if result == test.expected_result:
            print("Test passed for " + str(test.input.nums))
        else:
            print(f"Test failed for {test.input.nums}. Expected: {test.expected_result}, Actual: {result}")

In [49]:
tests = [
    Test(Input([3, 2, 2, 3, 4, 6, 7, 7, -1],4),20),
    Test(Input([5, 5, 5, 5, 5],3),0),
    Test(Input([2,2,2],3),0),
    Test(Input([-2,-2,-2],3),0),
    Test(Input([2,3,4],3),9),
    Test(Input([-2,2,2],2),0),
    Test(Input([-2,2,4],2),6),
    Test(Input([-2,-4,-6],3),-12),
    Test(Input([-2,-4,-8],2),-6),
    Test(Input( [11,11,7,2,9,4,17,1],1),17),
    Test(Input([9,9,9,1,2,3],3),12),
    #Test(Input(,)),
]

run_tests(tests, Solution().maxSum)

Test passed for [3, 2, 2, 3, 4, 6, 7, 7, -1]
Test passed for [5, 5, 5, 5, 5]
Test passed for [2, 2, 2]
Test passed for [-2, -2, -2]
Test passed for [2, 3, 4]
Test passed for [-2, 2, 2]
Test passed for [-2, 2, 4]
Test passed for [-2, -4, -6]
Test passed for [-2, -4, -8]
Test passed for [11, 11, 7, 2, 9, 4, 17, 1]
Test passed for [9, 9, 9, 1, 2, 3]
